# Point clouds

**Geometry type:** `point_cloud`

The lifecycle of a point cloud store:

1. Generate synthetic points with per-point attributes
2. Write them to a `.zv` store
3. Open the store and inspect its metadata
4. Read every point, then only the points in a bounding box
5. Build a pyramid and read its coarser levels
6. Export to CSV
7. Validate the store

Needs `zarr-vectors-tools` (which installs `zarr-vectors`) and `pandas`.

In [1]:
import os
import tempfile
from pathlib import Path

import numpy as np

# Every store in this notebook goes to a temporary directory.
_tmpdir = tempfile.mkdtemp(prefix="zv_examples_")
STORE = os.path.join(_tmpdir, "scan.zv")
print("Store path:", STORE)

Store path: /tmp/zv_examples_80c3x8gd/scan.zv


## 1 · Generate synthetic data

100,000 points spread uniformly through a 1000 µm cube, so neighbouring
points are about 21.5 µm apart (1000 µm / 100,000^(1/3)). Each point has a
float `intensity`, an integer tissue `label` (0–7) and a float `confidence`.

In [2]:
rng = np.random.default_rng(42)

N = 100_000
positions = rng.uniform(0, 1000, (N, 3)).astype(np.float32)   # µm

intensity  = rng.uniform(0, 1, N).astype(np.float32)
label      = rng.integers(0, 8, N).astype(np.int32)
confidence = rng.uniform(0.5, 1, N).astype(np.float32)

print(f"positions : {positions.shape}  dtype={positions.dtype}")
print(f"intensity : {intensity.shape}  range [{intensity.min():.3f}, {intensity.max():.3f}]")
print(f"label     : {label.shape}  classes {np.unique(label)}")
print(f"spacing   : {1000 / N ** (1 / 3):.1f} µm")

positions : (100000, 3)  dtype=float32
intensity : (100000,)  range [0.000, 1.000]
label     : (100000,)  classes [0 1 2 3 4 5 6 7]
spacing   : 21.5 µm


## 2 · Write the store

`chunk_shape` is the unit of storage: 250 µm chunks cut the cube into
4 × 4 × 4 = 64 chunks. `bin_shape` is the grid that bounding-box reads
target and that the pyramid merges points on. Set it near the point spacing,
here 25 µm, so that each coarser level merges a modest number of points per
bin. The default bin is the whole chunk, which would leave the first coarse
level (500 µm bins) with 8 points.

In [3]:
from zarr_vectors.types.points import write_points

summary = write_points(
    STORE,
    positions,
    chunk_shape=(250.0, 250.0, 250.0),
    bin_shape=(25.0, 25.0, 25.0),
    vertex_attributes={
        "intensity":  intensity,
        "label":      label,
        "confidence": confidence,
    },
)
print(summary)
print("Level 0 contents:", sorted(p.name for p in Path(STORE, "0").iterdir()))

{'vertex_count': 100000, 'chunk_count': 64, 'object_count': 0, 'group_count': 0, 'bins_per_chunk': (10, 10, 10)}
Level 0 contents: ['vertex_attributes', 'vertex_fragments', 'vertices', 'zarr.json']


## 3 · Open the store and inspect metadata

`zarr_vectors.open` returns a `Dataset`. Opening it and asking for counts,
bounds and grid shapes reads metadata only, not vertex data.

In [4]:
import zarr_vectors as zv

ds = zv.open(STORE)
print(ds)
print()
lv0 = ds.level(0)
print("kinds        :", ds.kinds)
print("ndim         :", ds.ndim)
print("levels       :", ds.levels)
print("chunk shape  :", lv0.scale)        # one grid cell (chunk) at this level
print("bin shape    :", lv0.resolution)   # one bin, the finest spatial-query cell
print("grid         :", lv0.grid)
print("bounds       :", ds.bounds[0].round(2), "→", ds.bounds[1].round(2))
print("attributes   :", lv0.attribute_names("vertex"))
print(f"vertex_count : {lv0.vertex_count:,}")

Dataset('file:///tmp/zv_examples_80c3x8gd/scan.zv', point_cloud, levels=[0])

kinds        : ('point_cloud',)
ndim         : 3
levels       : (0,)
chunk shape  : (250.0, 250.0, 250.0)
bin shape    : (25.0, 25.0, 25.0)
grid         : Grid(4x4x4 cells of (250.0, 250.0, 250.0))
bounds       : [0.03 0.01 0.01] → [1000.    999.99  999.97]
attributes   : ('confidence', 'intensity', 'label')
vertex_count : 100,000


## 4 · Read all points

In [5]:
from zarr_vectors.types.points import read_points

result = read_points(STORE, attribute_names=["intensity", "label", "confidence"])

print(f"vertex_count : {result['vertex_count']:,}")
print(f"positions    : {result['positions'].shape}  dtype={result['positions'].dtype}")
for name, values in result["vertex_attributes"].items():
    print(f"{name:<12} : {values.shape}  dtype={values.dtype}")
print()
print("First 3 positions (chunk order, not input order):")
print(result["positions"][:3])

vertex_count : 100,000
positions    : (100000, 3)  dtype=float32
intensity    : (100000,)  dtype=float32
label        : (100000,)  dtype=int32
confidence   : (100000,)  dtype=float32

First 3 positions (chunk order, not input order):
[[ 1.283508  24.254599  14.788155 ]
 [ 3.0815606  9.29687    1.8303767]
 [16.074093  16.540844   6.853385 ]]


## 5 · Bounding-box query

`read_points(bbox=...)` reads only the chunks that intersect the box and
returns the points inside it. A 200 µm cube is 0.8% of the volume, so about
800 of the 100,000 points should come back.

In [6]:
lo = np.array([100.0, 100.0, 100.0])
hi = np.array([300.0, 300.0, 300.0])

bbox_result = read_points(STORE, bbox=(lo, hi), attribute_names=["intensity"])

expected_fraction = ((hi - lo) / 1000).prod()
print(f"Query bbox : {lo} → {hi}")
print(f"Expected   : ~{int(N * expected_fraction):,} points ({expected_fraction * 100:.1f}% of the volume)")
print(f"Returned   : {bbox_result['vertex_count']:,} points")

assert (bbox_result["positions"] >= lo).all() and (bbox_result["positions"] <= hi).all()
print("All returned points lie inside the box.")

Query bbox : [100. 100. 100.] → [300. 300. 300.]
Expected   : ~800 points (0.8% of the volume)
Returned   : 797 points
All returned points lie inside the box.


## 6 · Build a pyramid

`build_pyramid(factors=[(coarsen, sparsity), ...])` adds one coarser level
per entry. Both factors are relative to the level below, so `(2, 1), (2, 1)`
gives bins 2× and 4× the 25 µm base bin (50 µm and 100 µm). Within each bin,
the points merge into one point at their mean position.

These points carry no object ids, so the whole cloud is one object and a
sparsity factor above 1 has nothing to drop: keep it at 1. To thin a cloud by
a random subset instead of by bin means, use
`zarr_vectors_tools.multiresolution.strategies.points.build_point_subset_pyramid`.
`cross_level_storage="none"` skips the links between levels, which the viewer
does not read.

In [7]:
from zarr_vectors_tools.multiresolution.coarsen import build_pyramid

info = build_pyramid(STORE, factors=[(2, 1), (2, 1)], cross_level_storage="none")
print(f"levels created: {info['levels_created']}  (method: {info['method']})")

ds = zv.open(STORE)   # reopen to see the new levels
for lv in ds.levels:
    level = ds.level(lv)
    print(f"  Level {lv}: {level.vertex_count:>7,} points  "
          f"bin shape={level.resolution}  chunk shape={level.scale}")

levels created: 2  (method: per_object)
  Level 0: 100,000 points  bin shape=(25.0, 25.0, 25.0)  chunk shape=(250.0, 250.0, 250.0)
  Level 1:   8,000 points  bin shape=(50.0, 50.0, 50.0)  chunk shape=(250.0, 250.0, 250.0)
  Level 2:   1,000 points  bin shape=(100.0, 100.0, 100.0)  chunk shape=(250.0, 250.0, 250.0)


Each coarse level has one point per occupied bin: (1000 / 50)³ = 8,000 and
(1000 / 100)³ = 1,000. The chunk shape stays 250 µm, because
`chunk_scale_factors` was not given.

## 7 · Read the coarse levels

Per-point attributes come along to the coarse levels. A float column takes
the mean of the merged points, so `intensity` narrows towards 0.5 as bins
grow; an integer column such as `label` takes the value of one of the merged
points, so it stays a valid class.

In [8]:
for lv in ds.levels:
    r = read_points(STORE, level=lv, attribute_names=["intensity", "label"])
    inten, lab = r["vertex_attributes"]["intensity"], r["vertex_attributes"]["label"]
    print(f"Level {lv}: {r['vertex_count']:>7,} points  "
          f"intensity [{inten.min():.3f}, {inten.max():.3f}]  "
          f"label dtype={lab.dtype} values={np.unique(lab).tolist()}")

coarse = read_points(STORE, level=ds.levels[-1])
print(f"\nLevel {ds.levels[-1]} positions span "
      f"{coarse['positions'].min(axis=0).round(1)} → {coarse['positions'].max(axis=0).round(1)}")

Level 0: 100,000 points  intensity [0.000, 1.000]  label dtype=int32 values=[0, 1, 2, 3, 4, 5, 6, 7]
Level 1:   8,000 points  intensity [0.167, 0.988]  label dtype=int32 values=[0, 1, 2, 3, 4, 5, 6, 7]
Level 2:   1,000 points  intensity [0.381, 0.581]  label dtype=int32 values=[0, 1, 2, 3, 4, 5, 6, 7]

Level 2 positions span [46.2 45.3 45.9] → [955.2 953.9 953.6]


## 8 · Select through the Dataset

`Dataset.select` takes the same filters as the read functions (`level`,
`bbox`, `objects`, `groups`, `attributes`) and returns a `ReadResult` with
`positions` and `attributes`. The box from section 5 gives the same points:

In [9]:
res = ds.select(bbox=(lo, hi), attributes=["intensity"]).read()
print(f"points in the box : {res.vertex_count:,}")
print(f"attributes        : {res.attributes.names()}")
print(f"mean intensity    : {res.attributes['intensity'].mean():.3f}")
assert res.vertex_count == bbox_result["vertex_count"]

points in the box : 797
attributes        : ('intensity',)
mean intensity    : 0.501


## 9 · Export to CSV

`export_csv` writes positions only unless `attribute_names` lists the columns
to add. Every value is written with six decimals, so `label` reads back as a
float.

In [10]:
import pandas as pd
from zarr_vectors_tools.convert.export.csv_points import export_csv

csv_path = os.path.join(_tmpdir, "scan_export.csv")
export_csv(STORE, csv_path, attribute_names=["intensity", "label"])

df = pd.read_csv(csv_path)
print(df.head())
print(f"\n{len(df):,} rows")

           x          y          z  intensity  label
0   1.283508  24.254599  14.788155   0.512268    6.0
1   3.081561   9.296870   1.830377   0.411061    6.0
2  16.074093  16.540844   6.853385   0.645933    4.0
3   5.323536   5.469347  67.811790   0.248908    0.0
4  11.137674  15.967889  50.147537   0.242375    3.0

100,000 rows


## 10 · Validate

`validate(path, level=n)` runs cumulative checks from 1 (structure) to 5
(the pyramid). The one warning, "Point cloud but links array exists", is a
false positive in the validator: it appears on every point cloud, including
one with no pyramid, and the store has no links.

In [11]:
from zarr_vectors.validate import validate

print(validate(STORE, level=5).summary())

Level 5 validation: PASS
  39 passed, 1 warnings, 0 errors
  WARN:  Point cloud but links array exists


## Summary

| Step | API |
|------|-----|
| Write | `write_points(path, positions, chunk_shape, bin_shape, vertex_attributes=...)` |
| Open | `zarr_vectors.open(path)` → `Dataset` |
| Metadata | `ds.kinds`, `ds.bounds`, `ds.level(n).vertex_count`, `.scale`, `.resolution` |
| Read all | `read_points(path, attribute_names=[...])` → `positions`, `vertex_attributes` |
| Bbox query | `read_points(path, bbox=(lo, hi))` |
| Pyramid | `build_pyramid(path, factors=[(2, 1), (2, 1)], cross_level_storage="none")` |
| Select | `ds.select(level=n, bbox=(lo, hi)).read()` → `ReadResult` |
| Export | `export_csv(path, out, attribute_names=[...])` |
| Validate | `validate(path, level=5)` |